**Synthetic Data Generator**

In [10]:
import gradio as gr
from transformers import pipeline
from openai import OpenAI
from dotenv import load_dotenv
import os

In [ ]:
#Gemma Setup
pipe = pipeline(
    "text-generation",
    model=r"E:\AI models\gemma-2-2b-it"
)

#Qwen Setup
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [12]:
system_prompt = """
You are a data generator.
Return ONLY valid JSON array.

Each object must represent a bank transaction with:
- name
- amount
- date
- type (deposit, withdrawal, transfer)
- currency
- description
"""

In [14]:
def generate_synthetic_data(user_prompt, n_records):

    gemma_prompt = f"""
Generate {n_records} synthetic bank transactions.

Format:
name, amount, date, type, currency, description

User request:
{user_prompt}
"""

    gemma_output = pipe(gemma_prompt, max_new_tokens=300)[0]["generated_text"]
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Convert this into clean JSON:\n\n{gemma_output}"
        }
    ]

    response = client.chat.completions.create(
        model="qwen/qwen3.7-plus",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
demo = gr.Interface(
    fn=generate_synthetic_data,
    inputs=[
        gr.Textbox(label="Description of data"),
        gr.Number(label="Number of records", value=5)
    ],
    outputs=gr.Textbox(label="Generated JSON"),
    title="Synthetic Bank Transaction Generator (Gemma + Qwen)",
    description="Uses open-source (Gemma) + frontier model (Qwen via OpenRouter)"
)

demo.launch()